# CUDA 13 × vLLM-Omni v0.24.0rc1 × VoxCPM2 (Colab T4) — Vietnamese TTS stability test

Goal: prove we can run a **CUDA 13 userspace runtime** on Colab T4 (driver 580 /
"CUDA Version: 13.0"), then serve a **Vietnamese-native, emotional, voice-cloning
TTS** natively under the **latest** vLLM-Omni — no downgrade to vllm 0.19.0, no
cu128/torch 2.10 compatibility hacks.

Why this notebook exists separately from the VieNeu one: VieNeu-TTS-v2 is pinned
to `vllm==0.19.0` (torch 2.10 / cu128) because that was the only way to dodge the
Colab cu12.8 userspace trap. Once we prove cu13 works on Colab with a *current*
vllm-omni, we can fork a **stable** recent vllm-omni and register VieNeu-TTS-v2
on top of it — instead of being stuck on a year-old rc1 fork.

Pipeline tested here (end-to-end):

```bash
vllm serve openbmb/VoxCPM2 --omni --port 8000 --trust-remote-code
```

then POST `/v1/audio/speech` with Vietnamese text.

**Why VoxCPM2**: Vietnamese is one of its 30 training languages; Apache-2.0;
voice cloning + voice design; ~8GB VRAM (fits T4); RTF ~0.13–0.30; single-stage
AR (simpler than diffusion / codec two-stage stacks). It does need the external
`voxcpm>=2.0.3` pip package (loaded via `VLLM_OMNI_VOXCPM_CODE_PATH` / sibling
src / pip — see `vllm_omni/model_executor/models/voxcpm2/voxcpm2_import_utils.py`).

**Secondary test** (later cell): `k2-fsa/OmniVoice` — 646 languages incl.
Vietnamese (8481 h), 0.6B diffusion backbone, RTF 0.025, Apache-2.0; lighter but
diffusion (less emotional control).

> If start-up or inference fails, the failure output is captured in the launch
> cell + `/tmp/vllm_serve.log`. Paste it back so the cu13 / serving wiring can
> be fixed before forking.

In [ ]:
# ============================================================
# 0. Preflight — environment detection (fail fast)
# ============================================================
import os, sys, subprocess, time, json, urllib.request

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

print("\u2713 Colab:", IN_COLAB)
print("\u2713 Python:", sys.version.split()[0])

# Show CUDA driver vs userspace BEFORE we touch anything.
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], text=True).strip()
    print("\u2713 GPU / driver:", smi)
except Exception as e:
    raise RuntimeError("nvidia-smi failed — this notebook needs a GPU runtime: Runtime > Change runtime type > T4 GPU. Err: " + str(e))

# Colab CUDA userspace probe: the image historically ships libcudart.so.12 only.
def find_cudart():
    hits = []
    for root in ["/usr/local/cuda", "/usr/local/cuda*", "/usr/lib", "/usr/lib/x86_64-linux-gnu", "/lib/x86_64-linux-gnu"]:
        import glob
        for d in glob.glob(root):
            for f in os.listdir(d) if os.path.isdir(d) else []:
                if f.startswith("libcudart.so"):
                    hits.append(os.path.join(d, f))
    return sorted(set(hits))

cudart = find_cudart()
print("\u2713 libcudart present on image (pre-install):")
for c in cudart[:8]:
    print("   ", c)
if not cudart:
    print("    <none found in standard paths — torch 2.11+cu130 will ship libcudart.so.13 via the cuda-toolkit pip package>")

PORT = 8000
MODEL_PRIMARY = "openbmb/VoxCPM2"      # Vietnamese-native, emotional, voice-cloning
MODEL_SECONDARY = "k2-fsa/OmniVoice"   # lighter, 646 langs incl Vietnamese
MODEL = MODEL_PRIMARY
VLLM_OMNI_REPO = "https://github.com/vllm-project/vllm-omni.git"   # UPSTREAM (not the justHman fork)
VLLM_OMNI_TAG = "v0.24.0rc1"   # latest; ships vllm==0.24.0 which pins torch==2.11.0 / cu13 stack
print("\u2713 model (primary):", MODEL)
print("\u2713 model (secondary):", MODEL_SECONDARY)
print("\u2713 vllm-omni:", VLLM_OMNI_REPO, "@", VLLM_OMNI_TAG)

In [ ]:
# ============================================================
# 1. (Optional) Mount Drive for HF cache persistence across sessions
# ============================================================
MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MOUNTED = True
        print("\u2713 Drive mounted")
    except Exception as e:
        print("\u26a0 Drive mount skipped:", e)

if MOUNTED:
    hf_cache = "/content/drive/MyDrive/hf_cache"
    os.makedirs(hf_cache, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache
    os.environ["TRANSFORMERS_CACHE"] = os.path.join(hf_cache, "transformers")
    os.environ["HF_HUB_CACHE"] = os.path.join(hf_cache, "hub")
    print("\u2713 HF_HOME =", hf_cache)
else:
    print("\u26a0 HF cache stays in-session (/root/.cache/huggingface).")

In [ ]:
# ============================================================
# 2. Install uv + cu130 torch stack + vllm 0.24.0 (pip-only, CUDA 13)
# ============================================================
# COLAB CUDA 13 STRATEGY (pip-only):
#   - torch 2.11.0+cu130 wheel (download.pytorch.org/whl/cu130/) declares
#     `cuda-toolkit[cublas,cudart,cufft,cufile,cupti,curand,cusolver,cusparse,
#        nvjitlink,nvrtc,nvtx]==13.0.2` on Linux. That cuda-toolkit PyPI
#     meta-package pulls cuda-cudart 13.x which SHIPS libcudart.so.13 via pip.
#     => no conda, no manual LD_LIBRARY_PATH needed; the +cu130 wheels are
#        self-contained for the CUDA 13 userspace.
#   - vllm 0.24.0 (released 2026-06-30) Requires-Dist pins
#     torch==2.11.0 / torchaudio==2.11.0 / torchvision==0.26.0 +
#     nvidia-cutlass-dsl[cu13]==4.5.2 → built for the cu13 stack.
#   - --torch-backend=cu130 makes uv resolve torch/torchaudio/torchvision from
#     the cu130 index (= +cu130 local-version wheels). We do NOT use `auto`,
#     and we do NOT downgrade to cu128/torch 2.10 here.
#
# We uninstall the Colab-preinstalled torch 2.11.0+cu128 first so the cu130
# wheels replace it cleanly (Colab preinstall is cu128, not cu130).
!pip install -q uv
!uv self update 2>/dev/null || true
!pip uninstall -y torch torchvision torchaudio vllm 2>/dev/null || true

# Install the cu130 stack + vllm 0.24.0 in one uv solve so the whole torch
# family is aligned to 2.11.0+cu130. flashinfer (vllm dep) resolves from PyPI.
!uv pip install --system --torch-backend=cu130 \
    torch==2.11.0 torchaudio==2.11.0 torchvision==0.26.0 vllm==0.24.0

# NOTE: do NOT `import torch` in this cell — the kernel may still hold a stale
# torch module from a pre-mount cell. We verify imports in the NEXT cell after
# the runtime settles. If a previous cell already imported torch, RUN CELL 4's
# kernel-restart instruction (Runtime > Restart session) before proceeding.
import importlib, sys
# Drop any cached torch so the cu130 install is seen fresh.
for _m in [m for m in list(sys.modules) if m.split('.')[0] in ('torch','vllm','torchaudio','torchvision')]:
    del sys.modules[_m]
import vllm, torch, torchaudio, torchvision
print("\u2713 vllm:", vllm.__version__, "| torch:", torch.__version__, "| torchaudio:", torchaudio.__version__, "| torchvision:", torchvision.__version__)
print("\u2713 torch.version.cuda =", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
assert vllm.__version__.startswith("0.24.0"), f"expected vllm 0.24.0, got {vllm.__version__}"
assert torch.__version__.startswith("2.11.0"), f"expected torch 2.11.0, got {torch.__version__}"
assert "+cu130" in torch.__version__, f"expected torch+cu130, got {torch.__version__}"
assert torch.cuda.is_available(), "CUDA not available — runtime must be T4 GPU"
print("\u2713 GPU:", torch.cuda.get_device_name(0))

## ⚠ STOP — Runtime > Restart session (one time)

If cell 2's asserts show `torch 2.11.0+cu128` (NOT `+cu130`) and `torch.version.cuda = 12.x`, the Jupyter kernel cached the **old** torch module before the cu130 install replaced the files on disk. This is the classic Colab `sys.modules` trap: the new wheel is installed but invisible to the running kernel.

**Fix: `Runtime > Restart session`, then re-run cell 2.** Do NOT skip this — a stale torch under a cu130 vllm is the most common silent break here. After restart, re-run cells 1–2 only (cell 1 re-mounts Drive / re-imports env); you do NOT need to re-run the install.

In [ ]:
# ============================================================
# 3. Clone UPSTREAM vllm-omni v0.24.0rc1 + install editable (cu13)
# ============================================================
# Clone the UPSTREAM vllm-omni at the latest tag (NOT the justHman/vieneu
# fork) so we test a clean cu13 baseline. Once this is stable, we fork
# v0.24.0rc1 and port the VieNeu registration onto it separately.
import os, subprocess, sys, importlib

!rm -rf /content/vllm-omni
!git clone --depth 1 --branch {VLLM_OMNI_TAG} {VLLM_OMNI_REPO} /content/vllm-omni
!cd /content/vllm-omni && git log --oneline -1

# Editable install of upstream vllm-omni into the cu13 stack. Pin cu130 so
# uv does not hop index. setup.py resolves platform deps dynamically
# (VLLM_OMNI_TARGET_DEVICE=cuda); it does NOT pin vllm itself, so the 0.24.0
# we just installed stays.
os.environ["VLLM_OMNI_TARGET_DEVICE"] = "cuda"
os.environ["UV_TORCH_BACKEND"] = "cu130"
!uv pip install --system -e /content/vllm-omni --torch-backend=cu130

# Install the external voxcpm package (VoxCPM2 native model loader). vLLM-Omni's
# voxcpm2_import_utils discovers it via env VLLM_OMNI_VOXCPM_CODE_PATH -> sibling
# ../VoxCPM/src -> pip-installed `voxcpm`. Pip is the no-clone path.
!uv pip install --system --torch-backend=cu130 "voxcpm>=2.0.3"

# The editable install wrote a .pth into site-packages, but THIS Jupyter kernel
# started before that .pth existed, so it will not re-scan site-packages until a
# restart. The `vllm` CLI subprocess (later cell) starts fresh and sees it fine;
# to verify imports HERE without a restart, prepend the source tree explicitly.
import sys as _sys
if "/content/vllm-omni" not in _sys.path:
    _sys.path.insert(0, "/content/vllm-omni")
import vllm_omni, voxcpm
print("\u2713 vllm_omni:", getattr(vllm_omni, "__version__", "<no __version__>"))
print("\u2713 voxcpm:", voxcpm.__version__ if hasattr(voxcpm, "__version__") else "importable")
from vllm_omni.config.pipeline_registry import OMNI_PIPELINES
print("\u2713 voxcpm2 registered in OMNI_PIPELINES:", "voxcpm2" in OMNI_PIPELINES)
from vllm_omni.model_executor.models.voxcpm2.voxcpm2_import_utils import import_voxcpm2_core
VoxCPM = import_voxcpm2_core()
print("\u2713 voxcpm.core.VoxCPM:", VoxCPM)

In [ ]:
# ============================================================
# 4. Pre-download the VoxCPM2 checkpoint (avoids first-serve timeout)
# ============================================================
from huggingface_hub import snapshot_download

print("Downloading", MODEL, "...")
mp = snapshot_download(MODEL)
print("\u2713 checkpoint at:", mp)

# Optional: pre-fetch the secondary OmniVoice checkpoint so the secondary test
# cell does not stall later. Skip if you only want VoxCPM2.
try:
    print("Downloading secondary", MODEL_SECONDARY, "...")
    snapshot_download(MODEL_SECONDARY)
    print("\u2713 secondary checkpoint cached")
except Exception as e:
    print("\u26a0 secondary pre-fetch skipped:", e)

In [ ]:
# ============================================================
# 5. Launch `vllm serve openbmb/VoxCPM2 --omni` in the background
# ============================================================
import subprocess, os, time

LOG = "/tmp/vllm_serve.log"
# Kill any stale server on PORT.
!fuser -k {PORT}/tcp 2>/dev/null || true

# --dtype=half: T4 is compute capability 7.5 and does NOT support bfloat16
# (needs >= 8.0). vllm_omni/deploy/voxcpm2.yaml hardcodes dtype: bfloat16
# (tuned for L4 cc8.9 / H20); CLI --dtype=half overrides the deploy yaml
# (caller-typed value > deploy YAML). float16 is the T4-correct substitute.
cmd = [
    "vllm", "serve", MODEL,
    "--omni",
    "--port", str(PORT),
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--dtype", "half",
]
print("Launching:", " ".join(cmd))
print("Logs ->", LOG)

logf = open(LOG, "w", buffering=1)
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd="/content/vllm-omni", env=dict(os.environ))
print("\u2713 server PID:", proc.pid)

In [ ]:
# ============================================================
# 6. Wait for /v1/models to come up (print log tail if it times out)
# ============================================================
import urllib.request, json, time

def tail(path, n=80):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-n:])
    except Exception as e:
        return f"<could not read {path}: {e}>"

def get_models(timeout=1500):
    url = f"http://localhost:{PORT}/v1/models"
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return json.loads(r.read())
        except Exception as e:
            last_err = e
        if proc.poll() is not None:
            raise RuntimeError(f"server process exited early (code {proc.returncode}). See {LOG}.\\n" + tail(LOG))
        time.sleep(5)
    raise RuntimeError(f"timed out waiting for {url}. last_err={last_err}. Tail of {LOG}:\\n" + tail(LOG))

print("Waiting for server (up to 25 min for first-time compile) ...")
models = get_models(timeout=1500)
print("\u2713 /v1/models:")
print(json.dumps(models, indent=2))

In [ ]:
# ============================================================
# 7. POST /v1/audio/speech (Vietnamese) and save a WAV
# ============================================================
import requests

TEXT = "Xin ch\u00e0o, \u0111\u00e2y l\u00e0 gi\u1ecdng n\u00f3i ti\u1ebfng Vi\u1ec7t t\u1eeb VoxCPM2 tr\u00ean vLLM-Omni."

def speech(text=TEXT, voice="default", response_format="wav"):
    payload = {"model": MODEL, "input": text, "voice": voice, "response_format": response_format}
    r = requests.post(f"http://localhost:{PORT}/v1/audio/speech", json=payload, timeout=180)
    if r.status_code != 200:
        raise RuntimeError(f"speech failed: {r.status_code} {r.text[:2000]}")
    return r

r = speech()
out_wav = "/content/voxcpm2_vi_demo.wav"
with open(out_wav, "wb") as f:
    f.write(r.content)
print("\u2713 wrote", out_wav, len(r.content), "bytes")

In [ ]:
# ============================================================
# 8. Play the generated audio inline
# ============================================================
from IPython.display import Audio, display
print("VoxCPM2 /v1/audio/speech (Vietnamese) output:")
display(Audio(out_wav, autoplay=False))

In [ ]:
# ============================================================
# 9. (Optional, secondary) OmniVoice — lighter 646-language TTS
# ============================================================
# Stop the VoxCPM2 server, relaunch OmniVoice (different model, same flag).
import subprocess, os, time

proc.terminate() if proc.poll() is None else None
time.sleep(5)
!fuser -k {PORT}/tcp 2>/dev/null || true

LOG2 = "/tmp/vllm_serve_omnivoice.log"
cmd2 = ["vllm", "serve", MODEL_SECONDARY, "--omni", "--port", str(PORT),
        "--host", "0.0.0.0", "--trust-remote-code"]
print("Launching secondary:", " ".join(cmd2))
logf2 = open(LOG2, "w", buffering=1)
proc2 = subprocess.Popen(cmd2, stdout=logf2, stderr=subprocess.STDOUT, cwd="/content/vllm-omni", env=dict(os.environ))
print("\u2713 server PID:", proc2.pid)

# Reuse the wait loop.
def get_models2(timeout=1500):
    url = f"http://localhost:{PORT}/v1/models"
    start = time.time(); last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200: return json.loads(r.read())
        except Exception as e: last_err = e
        if proc2.poll() is not None:
            raise RuntimeError(f"secondary exited early (code {proc2.returncode}).\\n" + tail(LOG2))
        time.sleep(5)
    raise RuntimeError(f"timed out. last_err={last_err}.\\n" + tail(LOG2))

print("Waiting for OmniVoice (up to 25 min) ...")
print(json.dumps(get_models2(), indent=2))

# Vietnamese TTS request to OmniVoice.
TEXT2 = "Xin ch\u00e0o, \u0111\u00e2y l\u00e0 gi\u1ecdng n\u00f3i t\u1eeb OmniVoice."
r2 = requests.post(f"http://localhost:{PORT}/v1/audio/speech",
                   json={"model": MODEL_SECONDARY, "input": TEXT2, "voice": "default", "response_format": "wav"},
                   timeout=180)
out2 = "/content/omnivoice_vi_demo.wav"
if r2.status_code == 200:
    open(out2, "wb").write(r2.content)
    print("\u2713 wrote", out2, len(r2.content), "bytes")
    display(Audio(out2, autoplay=False))
else:
    print("\u2717 OmniVoice speech failed:", r2.status_code, r2.text[:1500])

In [ ]:
# ============================================================
# 10. (Diagnostic) Show last 60 log lines of the active server
# ============================================================
active_log = LOG2 if (proc2.poll() is None or proc2.returncode is not None) else LOG
print(tail(active_log, n=60))

## Notes / known limitations / next step

- **CUDA 13 userspace via pip, no conda.** torch 2.11.0+cu130 declares
  `cuda-toolkit[cublas,cudart,...]==13.0.2` on Linux, and that PyPI
  meta-package pulls `cuda-cudart 13.x` which ships `libcudart.so.13` into
  `site-packages`. So the cu130 torch wheel is self-contained for the
  CUDA 13 userspace — Colab's image only having `libcudart.so.12` is no
  longer a blocker. This is the difference from the cu128/torch-2.10 path
  the VieNeu notebook is forced onto by `vllm==0.19.0`.

- **uv `--torch-backend=cu130` (NOT `auto`).** Colab driver 580 reports
  "CUDA Version: 13.0" via nvidia-smi (forward-compat driver); uv's `auto`
  reads the driver and also lands on cu130, but pinning it explicitly avoids
  any drift if uv's driver table changes. Requires a recent uv — cell 2 runs
  `uv self update`.

- **Kernel restart trap.** If any cell ran `import torch` before cell 2
  finished installing the cu130 wheels, Jupyter caches the old torch module
  and `torch.version.cuda` reads 12.x even though the cu130 files are on
  disk. Symptom: cell 2 asserts fail on `+cu130`. Fix: `Runtime >
  Restart session`, then re-run cells 1–2. This is documented above the
  restart marker cell.

- **VoxCPM2 external code.** vLLM-Omni's `voxcpm2_import_utils.py` finds the
  native model loader via `VLLM_OMNI_VOXCPM_CODE_PATH` → sibling
  `../VoxCPM/src` → pip `voxcpm`. We use the pip path
  (`voxcpm>=2.0.3`), which pulls modelscope/datasets/gradio as transitive
  deps — heavy-ish but only `voxcpm.core.VoxCPM` is actually imported by
  the serving path. If resolution is slow, pre-install in a Google Colab
  terminal with `uv pip install --system --torch-backend=cu130 voxcpm>=2.0.3`.

- **Why CosyVoice2/3, Qwen3-TTS, Voxtral are NOT tested here: none support
  Vietnamese.** CosyVoice2/3: 9 langs (zh/en/ja/ko/de/es/fr/it/ru). Qwen3-TTS:
  10 langs, no vi. Voxtral-TTS: 9 langs, no vi, and CC-BY-NC-4.0. VoxCPM2
  (30 langs incl vi, Apache-2.0) and OmniVoice (646 langs incl vi, Apache-2.0)
  are the only vllm-omni-native TTS that cover Vietnamese, so they are this
  notebook's targets. (VieNeu-TTS-v2 remains out-of-tree until ported.)

- **T4 (compute capability 7.5) constraints.** No FP8; **bfloat16 needs
  compute capability >= 8.0, so T4 cannot run bf16** — it errors with
  `ValueError: Bfloat16 is only supported on GPUs with compute capability of
  at least 8.0`. `vllm_omni/deploy/voxcpm2.yaml` hardcodes `dtype: bfloat16`
  (tuned for L4 cc8.9 / H20), so the VoxCPM2 launch MUST pass `--dtype half`
  on the CLI to override the deploy yaml (caller-typed > deploy YAML per
  `_create_from_registry`). float16 is the T4-correct substitute. OmniVoice's
  `deploy/omnivoice.yaml` uses `dtype: float32` (T4-safe), so the secondary
  cell needs no `--dtype` override. 16GB VRAM: VoxCPM2 ~8GB fits; OmniVoice
  diffusion heavier on first compile but fits at the stage's 0.5 util default.

- **Editable install + Jupyter kernel.** `uv pip install -e` writes a `.pth`
  into `site-packages`, but the kernel that started before the install will
  NOT re-scan `site-packages` until a restart. The `vllm` CLI subprocess
  (launched later in its own fresh process) sees the editable install fine,
  which is why `vllm serve` resolves `vllm_omni` even when
  `import vllm_omni` in the clone cell raised `ModuleNotFoundError`. To verify
  imports in the clone cell without a restart, that cell prepends
  `/content/vllm-omni` to `sys.path` explicitly before importing. If you
  restart the session after the install, you can drop that line — the `.pth`
  is then already on the path.

- **First launch** JIT-compiles a few CUDA kernels; the wait cell allows
  up to 25 min. The first `import transformers` inside the `vllm` CLI is
  slow (~30–60s) as it scans every model file — do NOT interrupt it
  (a KeyboardInterrupt during that import looks like a crash but isn't).
  Subsequent restarts are faster if Drive cache is mounted.

- **After this is green**, the plan is: fork `vllm-project/vllm-omni@v0.24.0rc1`
  (call it `feat/vieneu-tts-v2-on-v0.24`) and port the VieNeu registration
  (`model_executor/models/vieneu/` + the `vieneu` extra in `pyproject.toml`
  + stage config) onto it, so VieNeu rides the stable cu13 stack instead of
  the old v0.19.0rc1 / cu128 one.